# 04 — Phase 4 : générer un plan de jeu avec Claude

Objectif : transformer l'attribution SHAP (phase 3) en plan de jeu rédigé, pour une composition hypothétique (deux équipes de 5 champions, pas un match réel).

Toute la logique de classement (`composition_fact_ranking`) a été **industrialisée** dans `lol_assistant/game_plan.py` — ce notebook l'utilise, il ne la redéfinit pas. C'est exactement pour ça qu'on avait fait ce travail d'industrialisation en phase 2 : la logique réutilisable vit dans le package, les notebooks orchestrent et expliquent.

## Concept 1 — Classer les faits pour les deux équipes

`load_ranking_artifacts` lit les deux CSV produits par le notebook 3 (`global_shap_importance.csv`, `champion_feature_interactions.csv`) — aucun appel au modèle XGBoost ni à SHAP ici, juste une lecture de fichiers, donc très rapide. `composition_fact_ranking` applique la même logique que dans le notebook 3 (le max par champion, pas la somme).

In [ ]:
from lol_assistant.game_plan import load_ranking_artifacts, composition_fact_ranking

global_importance, interactions = load_ranking_artifacts()

your_team = ["Jayce", "Sylas", "Yone", "Yunara", "Lulu"]
enemy_team = ["Garen", "Rengar", "Akali", "Hwei", "Rakan"]

your_ranking = composition_fact_ranking(your_team, global_importance, interactions)
enemy_ranking = composition_fact_ranking(enemy_team, global_importance, interactions)

your_ranking.head(6)[["feature", "key_champion", "team_importance"]]

## Concept 2 — Construire le prompt

`build_game_plan_prompt` transforme les deux classements en texte : les faits de **notre** équipe ("à prioriser") et ceux de l'équipe **adverse** ("à surveiller/contrer" — même analyse légitime appliquée à leurs champions, pas un vrai signal de matchup entraîné, comme convenu). On limite volontairement aux 6 faits les plus importants par équipe (`top_n`) — inutile de noyer le LLM avec les 27, les derniers ont une importance quasi nulle de toute façon.

In [ ]:
from lol_assistant.game_plan import build_game_plan_prompt

prompt = build_game_plan_prompt(your_team, enemy_team, your_ranking, enemy_ranking)
print(prompt)

## Concept 3 — Appeler Claude

`generate_game_plan` fait tout le pipeline (classement + prompt + appel API) en un appel. On utilise **Claude Sonnet 5** par défaut (`DEFAULT_MODEL` dans `game_plan.py`) — tâche de rédaction assez "gabarisée" (données structurées → texte), pas besoin du modèle le plus capable ; passer `model="claude-opus-5"` pour la meilleure qualité possible si besoin.

**Nécessite `ANTHROPIC_API_KEY` renseignée dans `.env`** (comme pour Riot en phase 1).

In [ ]:
from lol_assistant.game_plan import generate_game_plan

game_plan_text, your_ranking, enemy_ranking = generate_game_plan(your_team, enemy_team)
print(game_plan_text)

---

**Checkpoint** : confirme que le plan de jeu généré est cohérent (mentionne bien les faits clés et les champions associés, pas de contresens). Prochaine étape : ajouter Ollama/Llama 3.1 comme option de comparaison (gratuite, locale).